# Estrategia Momentum (MSCI-like)

Implementacion del algoritmo descrito en el enunciado: momentum 12M y 6M con lag de 1 mes, normalizacion Z-score por mes, score compuesto, seleccion y pesos.


In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 50)


In [71]:

BACKTEST_START = pd.Timestamp('2015-01-31')

# Usamos historial amplio para poder calcular R_12 y R_6 con lag en enero-2015.
try:
    df = pd.read_parquet('sp500_history_copy.parquet')
    source_file = 'sp500_history_copy.parquet'
except Exception:
    df = pd.read_pickle('sp500_history_filtered.pkl')
    source_file = 'sp500_history_filtered.pkl'

df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['symbol', 'date']).reset_index(drop=True)
print('Fuente:', source_file)
print('Rango de fechas:', df['date'].min().date(), '->', df['date'].max().date())


Paso A: Retorno Acumulado con "Lag" de 1 Mes 
Se deben calcular dos variables de retorno para cada uno de los 10 activos (9 sectores + GLD) -utilizaremos retornos logarítmos-: 
1.  Momentum 12 Meses (R_12): Rentabilidad desde el mes t-13 al mes t-1. 
2.  Momentum 6 Meses (R_6): Rentabilidad desde el mes t-7 al mes t-1. 
(Nota: El mes t-1 es el mes anterior al rebalanceo; se excluye el mes actual para 
evitar el ruido de la reversión a la media).

In [72]:
# Resample mensual: ultimo dia habil con precio disponible por simbolo
monthly = (
    df.set_index('date')
      .groupby('symbol')
      .resample('ME')
      .last()
      .reset_index()
)

# Retornos logaritmicos mensuales por simbolo
monthly['log_ret'] = np.log(monthly['close'] / monthly.groupby('symbol')['close'].shift(1))
monthly = monthly.dropna(subset=['log_ret'])
monthly.head()


,symbol,date,assetid,security_name,sector,industry,subsector,in_sp500,open,high,low,close,volume,unadjusted_close,log_ret
1,A,2015-02-28,131684.0,Agilent Technologies Inc Common,Health Care,Life Sciences Tools & Services,Life Sciences Tools & Services,1.0,38.644352,38.699165,38.411388,38.562126,1834929.250,42.209999,0.111142
2,A,2015-03-31,131684.0,Agilent Technologies Inc Common,Health Care,Life Sciences Tools & Services,Life Sciences Tools & Services,1.0,37.805389,38.263302,37.642372,38.052662,1811670.500,41.549999,-0.013300
3,A,2015-04-30,131684.0,Agilent Technologies Inc Common,Health Care,Life Sciences Tools & Services,Life Sciences Tools & Services,1.0,38.309097,38.409836,37.649700,37.887814,1884973.625,41.369999,-0.004342
4,A,2015-05-31,131684.0,Agilent Technologies Inc Common,Health Care,Life Sciences Tools & Services,Life Sciences Tools & Services,1.0,38.272461,38.309097,37.384109,37.722965,5484166.000,41.189999,-0.004360
5,A,2015-06-30,131684.0,Agilent Technologies Inc Common,Health Care,Life Sciences Tools & Services,Life Sciences Tools & Services,1.0,35.880157,35.981148,35.329285,35.421097,3308319.000,38.580002,-0.062962


In [73]:

# Universo completo: todos los symbols del dataframe mensual
universe_ret = monthly[['date', 'sector', 'symbol', 'log_ret']].copy()
universe_ret['sector'] = universe_ret['sector'].fillna('Unknown')
universe_ret = universe_ret.sort_values(['symbol', 'date']).reset_index(drop=True)

print('Numero de symbols en el universo:', universe_ret['symbol'].nunique())
print('Rango del universo:', universe_ret['date'].min().date(), '->', universe_ret['date'].max().date())
universe_ret.head()


,sector,date,log_ret
0,Consumer Discretionary,2015-02-28,0.061341
1,Consumer Discretionary,2015-03-31,-0.000943
2,Consumer Discretionary,2015-04-30,-0.023716
3,Consumer Discretionary,2015-05-31,0.000280
4,Consumer Discretionary,2015-06-30,-0.015279


In [74]:

# Mantener meses con al menos TOP_K activos elegibles
TOP_K = 20
assets_per_month = universe_ret.groupby('date')['symbol'].nunique()
valid_dates = assets_per_month[assets_per_month >= TOP_K].index

universe_ret = universe_ret[universe_ret['date'].isin(valid_dates)].copy()

print('Meses validos (>= 20 symbols):', len(valid_dates))
print('Primer mes valido:', valid_dates.min().date())
print('Ultimo mes valido:', valid_dates.max().date())
print('Min symbols/mes:', assets_per_month.loc[valid_dates].min())
print('Max symbols/mes:', assets_per_month.loc[valid_dates].max())


,sector,symbol,date,log_ret
0,Consumer Discretionary,Consumer Discretionary,2015-02-28,0.061341
1,Consumer Discretionary,Consumer Discretionary,2015-03-31,-0.000943
2,Consumer Discretionary,Consumer Discretionary,2015-04-30,-0.023716
3,Consumer Discretionary,Consumer Discretionary,2015-05-31,0.000280
4,Consumer Discretionary,Consumer Discretionary,2015-06-30,-0.015279


In [75]:

print('Universo listo para momentum. Filas:', len(universe_ret))
print('Symbols unicos:', universe_ret['symbol'].nunique())
universe_ret.head()


GLD no encontrado en el parquet, se descargara desde Yahoo Finance
Universo de retornos creado con los sectores y GLD: sector
Consumer Discretionary    132
Consumer Staples          132
Energy                    132
Financials                132
GLD                       132
Health Care               132
Industrials               132
Information Technology    132
Materials                 132
Utilities                 132
Name: count, dtype: int64


,sector,symbol,date,log_ret
0,Consumer Discretionary,Consumer Discretionary,2015-02-28,0.061341
1,Consumer Discretionary,Consumer Discretionary,2015-03-31,-0.000943
2,Consumer Discretionary,Consumer Discretionary,2015-04-30,-0.023716
3,Consumer Discretionary,Consumer Discretionary,2015-05-31,0.000280
4,Consumer Discretionary,Consumer Discretionary,2015-06-30,-0.015279


In [76]:

# Paso A: Momentum 12M y 6M con lag de 1 mes (se excluye el mes actual)
def add_momentum(df_in, window):
    col = f'R_{window}'
    df_in[col] = (
        df_in.groupby('symbol')['log_ret']
            .transform(lambda s: s.shift(1).rolling(window=window, min_periods=window).sum())
    )
    return df_in

mom = universe_ret.copy()
mom = add_momentum(mom, 12)
mom = add_momentum(mom, 6)
mom = mom.dropna(subset=['R_12', 'R_6']).copy()

# Backtest mensual desde fin de enero 2015
mom = mom[mom['date'] >= BACKTEST_START].copy()
rebalance_dates = pd.DatetimeIndex(sorted(mom['date'].unique()))

if BACKTEST_START not in rebalance_dates:
    first_available = rebalance_dates.min().date() if len(rebalance_dates) > 0 else 'N/A'
    raise ValueError(
        f'No hay datos suficientes para iniciar en {BACKTEST_START.date()} con lag de 1 mes y ventanas 12/6. '
        f'Primer mes disponible: {first_available}'
    )

expected_months = pd.date_range(start=BACKTEST_START, end=rebalance_dates.max(), freq='ME')
missing_months = expected_months.difference(rebalance_dates)
if len(missing_months) > 0:
    print('Advertencia: faltan meses de rebalanceo:', len(missing_months))

print('Primer rebalanceo:', rebalance_dates.min().date())
print('Ultimo rebalanceo:', rebalance_dates.max().date())
print('Numero de rebalanceos:', len(rebalance_dates))
mom.head()


,sector,symbol,date,log_ret,R_12,R_6
12,Consumer Discretionary,Consumer Discretionary,2016-02-29,0.043403,-0.130119,-0.171807
13,Consumer Discretionary,Consumer Discretionary,2016-03-31,0.079784,-0.148057,-0.083955
14,Consumer Discretionary,Consumer Discretionary,2016-04-30,-0.031312,-0.067330,0.036559
15,Consumer Discretionary,Consumer Discretionary,2016-05-31,-0.033534,-0.074926,-0.034001
16,Consumer Discretionary,Consumer Discretionary,2016-06-30,-0.013127,-0.108740,-0.035475


Paso B: Normalización por Factor (Z-Score)

In [77]:
# Paso B: Z-score por mes
for col in ['R_12', 'R_6']:
    mu = mom.groupby('date')[col].transform('mean')
    sigma = mom.groupby('date')[col].transform('std')
    suffix = col.split('_')[1]
    mom[f'Z_{suffix}'] = (mom[col] - mu) / sigma

# Paso C: Score final y seleccion
mom['score'] = (mom['Z_12'] + mom['Z_6']) / 2
mom.head()


,sector,symbol,date,log_ret,R_12,R_6,Z_12,Z_6,score
12,Consumer Discretionary,Consumer Discretionary,2016-02-29,0.043403,-0.130119,-0.171807,-0.019316,-0.404888,-0.212102
13,Consumer Discretionary,Consumer Discretionary,2016-03-31,0.079784,-0.148057,-0.083955,-0.015700,-0.164866,-0.090283
14,Consumer Discretionary,Consumer Discretionary,2016-04-30,-0.031312,-0.067330,0.036559,-0.047964,-0.325924,-0.186944
15,Consumer Discretionary,Consumer Discretionary,2016-05-31,-0.033534,-0.074926,-0.034001,-0.203137,-0.834110,-0.518623
16,Consumer Discretionary,Consumer Discretionary,2016-06-30,-0.013127,-0.108740,-0.035475,-0.388844,-0.886927,-0.637885


Paso C: Puntuación Compuesta y Selección

In [78]:
# Seleccion top 20 (o menos si no hay 20 activos) y pesos
TOP_K = 20
FIXED_WEIGHT = 0.05  # 5% por activo seleccionado

def select_top(g):
    g = g.sort_values('score', ascending=False)
    top_k = min(TOP_K, len(g))
    top = g.head(top_k).copy()
    top['weight'] = FIXED_WEIGHT
    top['cash_weight'] = 1 - FIXED_WEIGHT * top_k
    return top

selection = mom.groupby('date', group_keys=True).apply(select_top)
if 'date' not in selection.columns:
    selection = selection.reset_index()
   
selection = selection[['date', 'sector', 'symbol', 'score', 'weight']].copy()
print('Activos seleccionados (primeras filas):')
print(selection.head(20))
selection.head()


Activos seleccionados (primeras filas):
         date                  sector                  symbol     score  weight
0  2016-02-29        Consumer Staples        Consumer Staples  1.104746    0.05
1  2016-02-29               Utilities               Utilities  0.929978    0.05
2  2016-02-29                     GLD                     GLD  0.574473    0.05
3  2016-02-29             Health Care             Health Care  0.303055    0.05
4  2016-02-29  Information Technology  Information Technology  0.088959    0.05
5  2016-02-29              Financials              Financials -0.006255    0.05
6  2016-02-29             Industrials             Industrials -0.045396    0.05
7  2016-02-29  Consumer Discretionary  Consumer Discretionary -0.212102    0.05
8  2016-02-29               Materials               Materials -0.492931    0.05
9  2016-02-29                  Energy                  Energy -2.244528    0.05
10 2016-03-31               Utilities               Utilities  0.973846    0.05


,date,sector,symbol,score,weight
0,2016-02-29,Consumer Staples,Consumer Staples,1.104746,0.05
1,2016-02-29,Utilities,Utilities,0.929978,0.05
2,2016-02-29,GLD,GLD,0.574473,0.05
3,2016-02-29,Health Care,Health Care,0.303055,0.05
4,2016-02-29,Information Technology,Information Technology,0.088959,0.05


In [79]:

# Open/Close mensuales por symbol
all_prices = monthly[['date', 'symbol', 'open', 'close']].copy()
all_prices['open'] = pd.to_numeric(all_prices['open'], errors='coerce')
all_prices['close'] = pd.to_numeric(all_prices['close'], errors='coerce')
all_prices = all_prices.dropna(subset=['open', 'close'])
all_prices = all_prices.sort_values(['symbol', 'date'])
all_prices.head()


,sector,symbol,date,open,close
0,Consumer Discretionary,Consumer Discretionary,2015-02-28,79.947540,79.832130
1,Consumer Discretionary,Consumer Discretionary,2015-03-31,80.021690,80.014038
2,Consumer Discretionary,Consumer Discretionary,2015-04-30,79.573029,78.939438
3,Consumer Discretionary,Consumer Discretionary,2015-05-31,79.766068,78.980614
4,Consumer Discretionary,Consumer Discretionary,2015-06-30,78.652466,78.481453


In [80]:

# Anexar open/close a la seleccion
selection = selection.merge(
    all_prices,
    on=['date', 'symbol'],
    how='left',
)
selection.head()


,date,sector,symbol,score,weight,open,close
0,2016-02-29,Consumer Staples,Consumer Staples,1.104746,0.05,52.800476,52.681828
1,2016-02-29,Utilities,Utilities,0.929978,0.05,33.701042,33.785152
2,2016-02-29,GLD,GLD,0.574473,0.05,117.589996,118.639999
3,2016-02-29,Health Care,Health Care,0.303055,0.05,83.880875,82.708946
4,2016-02-29,Information Technology,Information Technology,0.088959,0.05,37.589130,37.503788


In [81]:
# Exportar seleccion mensual a CSV (incluye symbol y open/close)
selection_out = selection[['date', 'sector', 'symbol', 'score', 'weight', 'open', 'close']].copy()
selection_out.to_csv('seleccion_momentum.csv', index=False)
print('CSV generado:', 'seleccion_momentum.csv', 'filas:', len(selection_out))


CSV generado: seleccion_momentum.csv filas: 1200
